In [7]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from scipy.sparse import hstack

In [8]:
train = pd.read_csv("salary-train.csv")
test = pd.read_csv("salary-test-mini.csv")

Предобработка текста(приведение к нижнему регистру, замена всего, кроме букв и цифр, на пробелы)

In [9]:
train['FullDescription'] = train['FullDescription'].apply(
    lambda x: re.sub('[^a-zA-Z0-9]', ' ', x.lower()) if isinstance(x, str) else ''
)
test['FullDescription'] = test['FullDescription'].apply(
    lambda x: re.sub('[^a-zA-Z0-9]', ' ', x.lower()) if isinstance(x, str) else ''
)

преобразование текстов в векторы признаков.
Оставили только те слова, которые встречаются хотя бы в 5 объектах (параметр min_df у TfidfVectorizer)

In [10]:
tfidf = TfidfVectorizer(min_df=5)
X_train_text = tfidf.fit_transform(train['FullDescription'])
X_test_text = tfidf.transform(test['FullDescription'])

Замена пропусков в столбцах LocationNormalized и ContractTime на специальную строку 'nan'

In [11]:
train['LocationNormalized'] = train['LocationNormalized'].fillna('nan')
train['ContractTime'] = train['ContractTime'].fillna('nan')
test['LocationNormalized'] = test['LocationNormalized'].fillna('nan')
test['ContractTime'] = test['ContractTime'].fillna('nan')


Применили DictVectorizer для получения one-hot-encoding признаков LocationNormalized и ContractTime.
Объединили все полученные признаки в одну матрицу "объекты-признаки".

In [12]:
enc = DictVectorizer()
X_train_categ = enc.fit_transform(
    train[['LocationNormalized', 'ContractTime']].to_dict('records'))

X_test_categ = enc.transform(
    test[['LocationNormalized', 'ContractTime']].to_dict('records'))

X_train = hstack([X_train_text, X_train_categ])
X_test = hstack([X_test_text, X_test_categ])


Постройте прогнозы для двух примеров из файла salary-test-mini.csv.
Значения полученных прогнозов являются ответом на задание. Укажите их через пробел

In [16]:
target = train['SalaryNormalized']
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, target)

predictions = ridge.predict(X_test)

print(round(predictions[0], 2), round(predictions[1], 2))

56577.28 37200.19
